# Cointegration Strategy Testing Framework



This notebook tests baskets for cointegration sustainability and mean reversion speed.



## Testing Pipeline (with Data Splitting to Prevent Overfitting):

1. **Load price data** from parquet files

2. **Split data chronologically**:

   - Cluster data: Most recent 20% (1.2-0.8 quantile) - for basket generation

   - Cointegration data: Middle 60% (0.8-0.2 quantile) - for cointegration testing

   - Half-life data: Oldest 20% (0.2-0.0 quantile) - for mean reversion testing

3. **Generate candidate baskets** using cluster data (prevents overfitting)

4. **Test initial cointegration** using cointegration data (auto-deduplicates overlapping baskets)

5. **Filter by sustainability** using cointegration data (rolling windows + discrete periods)

6. **Filter by mean reversion speed** using half-life data (prevents data leakage)

7. **Output validated baskets** for strategy deployment



## Why Data Splitting?

- **Prevents overfitting**: Clustering on same data as testing causes false positives

- **Prevents look-ahead bias**: Using future data to select baskets

- **Prevents data leakage**: Testing on data used for training

- **Realistic validation**: Tests baskets on unseen data


In [1]:
import sys
from pathlib import Path
import logging
from datetime import datetime, timedelta, timezone

# Find workspace root
current = Path().resolve()
while current.name != 'hku-datawork' and current.parent != current:
    current = current.parent
workspace_root = current if current.name == 'hku-datawork' else Path('/home/leo/GitHub_Clones/hku_stuff/hku-datawork')
sys.path.insert(0, str(workspace_root))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.data_split import split_data_chronologically
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion
from hypothesis_testing.cointegration.deduplicate_baskets import filter_overlapping_baskets

logger.info("Imports completed")

2025-11-06 13:48:36,053 - __main__ - INFO - Imports completed


## Configuration

Set data paths, timeframes, and filter thresholds.

In [2]:
# Configuration
DATA_DIR = Path("/workspace-hku/hku-data/test_data")  # Use absolute path to actual data location
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'  # 15m only available as 'mark', 1h only as 'index'
SYMBOLS = None  # None = auto-discover all symbols
# Discover available symbols from parquet files in data directory
pattern = f"*_{PRICE_TYPE}_{TIMEFRAME}_*.parquet"
parquet_files = list(DATA_DIR.glob(pattern))
available_symbols = set()
for f in parquet_files:
    # Pattern: perpetual_{SYMBOL}_mark_15m_...
    parts = f.stem.split('_')
    if len(parts) >= 2:
        symbol = parts[1]  # Extract symbol (e.g., "BTC" or "1000BONK")
        available_symbols.add(symbol)

AVAILABLE_SYMBOLS = sorted(list(available_symbols))
print(f"Found {len(AVAILABLE_SYMBOLS)} symbols available in {DATA_DIR}")

# Basket generation
MIN_BASKET_SIZE = 3
MAX_BASKET_SIZE = 6
N_CLUSTERS = 10


# Parallel processing
MAX_WORKERS = None  # None = use CPU count

# Bars per day (depends on timeframe)
BARS_PER_DAY = 96


Found 116 symbols available in /workspace-hku/hku-data/test_data


## Step 1: Load Price Data

Load historical price data from parquet files.

In [3]:
# Load 15-minute mark price data from existing parquet files
# Data is already available in DATA_DIR - no download needed
logger.info(f"Loading data from parquet files in {DATA_DIR}")
logger.info(f"Timeframe: {TIMEFRAME}, Price type: {PRICE_TYPE}")

# Load data using the existing data loader (loads from parquet files)
# SYMBOLS=None means it will discover and load all symbols automatically
price_data = load_price_data(
    data_dir=DATA_DIR,
    years_back=1.2,  # Load only last 2 years
    symbols=None,  # Auto-discover all symbols from parquet files
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=MAX_WORKERS
)

print(f"  Symbols: {len(price_data.columns)}")
print(f"  Timestamps: {len(price_data):,}")
print(f"  Date range: {price_data.index.min()} to {price_data.index.max()}")


2025-11-06 13:48:36,086 - __main__ - INFO - Loading data from parquet files in /workspace-hku/hku-data/test_data
2025-11-06 13:48:36,088 - __main__ - INFO - Timeframe: 15m, Price type: mark
2025-11-06 13:48:36,091 - hypothesis_testing.cointegration.data_loader - INFO - Loading data from 116 parquet files for 116 symbols...
2025-11-06 13:48:37,735 - hypothesis_testing.cointegration.data_loader - INFO - Aligning timestamps across all symbols...
2025-11-06 13:48:38,438 - hypothesis_testing.cointegration.data_loader - INFO - Loaded 34939 timestamps for 104 symbols
2025-11-06 13:48:38,440 - hypothesis_testing.cointegration.data_loader - INFO - Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


  Symbols: 104
  Timestamps: 34,939
  Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


## Step 2: Split Data Chronologically

**CRITICAL**: Split data to prevent overfitting and data leakage:
- **Cluster data** (1.2-0.8): Most recent 20% - used for basket generation
- **Cointegration data** (0.8-0.2): Middle 60% - used for cointegration testing
- **Half-life data** (0.2-0.0): Oldest 20% - used for mean reversion testing

This ensures we don't test on the same data we used to generate baskets.

In [4]:
# Split data chronologically to prevent overfitting
splits = split_data_chronologically(
    price_data,
    cluster_split=(1.2, 0.8),      # Most recent 20% for clustering
    cointegration_split=(0.8, 0.4),
    test_split=(0.4, 0.0) 
)

cluster_data = splits['cluster_data']
cointegration_data = splits['cointegration_data']
test_data = splits['test_data']



2025-11-06 13:48:38,479 - hypothesis_testing.cointegration.data_split - INFO - Split data chronologically:
2025-11-06 13:48:38,481 - hypothesis_testing.cointegration.data_split - INFO -   Cluster: 6987 samples (1.2-80%) from 2025-07-01 16:15:00+00:00 to 2025-09-12 10:45:00+00:00
2025-11-06 13:48:38,482 - hypothesis_testing.cointegration.data_split - INFO -   Cointegration: 13976 samples (80%-40%) from 2025-02-06 02:15:00+00:00 to 2025-07-01 16:00:00+00:00
2025-11-06 13:48:38,483 - hypothesis_testing.cointegration.data_split - INFO -   Test: 13976 samples (40%-0%) from 2024-09-13 12:15:00+00:00 to 2025-02-06 02:00:00+00:00


## Step 3: Generate Candidate Baskets

Generate candidate baskets using clustering on **cluster data only** (prevents overfitting).

In [5]:
# Extract base symbols (remove _close suffix)

base_symbols = [col.replace('_close', '') for col in price_data.columns]



candidate_baskets = generate_baskets_clustering(

    price_data=cluster_data,  # Use cluster_data, not price_data!

    n_clusters=N_CLUSTERS,

    min_basket_size=MIN_BASKET_SIZE,

    max_basket_size=MAX_BASKET_SIZE,

    parallel=True,

    max_workers=MAX_WORKERS

)



logger.info(f"Generated {len(candidate_baskets)} candidate baskets")

print(f"Sample baskets: {candidate_baskets[:5]}")

2025-11-06 13:48:38,916 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 14 to 12 symbols (estimated 6370 -> ~4004 combinations)
2025-11-06 13:48:38,985 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 49 to 12 symbols (estimated 16121000 -> ~4004 combinations)
2025-11-06 13:48:39,012 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 31 to 12 symbols (estimated 942152 -> ~4004 combinations)
2025-11-06 13:48:39,596 - __main__ - INFO - Generated 7293 candidate baskets


Sample baskets: [['GALA', 'THETA', 'GRT'], ['GALA', 'THETA', 'NEAR'], ['GALA', 'THETA', 'DOT'], ['GALA', 'THETA', 'RENDER'], ['GALA', 'THETA', 'ICP']]


## Step 4: Test Initial Cointegration

Test all candidate baskets for cointegration using Johansen test on **cointegration data** (separate from cluster data).
Automatically deduplicates baskets with >50% symbol overlap (keeps lowest p-value).

In [ ]:
cointegrated_baskets = test_baskets_cointegration_parallel(
    price_data=cointegration_data,  # Use cointegration_data, not price_data!
    candidate_baskets=candidate_baskets,
    max_workers=MAX_WORKERS,
    batch_size=100,
    overlap_threshold=0.5,
    prefer_lower_pvalue=True,
)

logger.info(
    f"Found {len(cointegrated_baskets)} unique cointegrated baskets "
    f"(after dedup) out of {len(candidate_baskets)} candidates"
)
print(
    f"Cointegration rate (after dedup): "
    f"{len(cointegrated_baskets)/len(candidate_baskets):.1%}"
)


2025-11-06 13:48:45,342 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 10/73 batches (1000 baskets), found 979 cointegrated so far
2025-11-06 13:48:47,567 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 20/73 batches (2000 baskets), found 1963 cointegrated so far
2025-11-06 13:48:49,497 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 30/73 batches (3000 baskets), found 2956 cointegrated so far
2025-11-06 13:48:51,527 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 40/73 batches (4000 baskets), found 3956 cointegrated so far
2025-11-06 13:48:53,414 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 50/73 batches (5000 baskets), found 4940 cointegrated so far
2025-11-06 13:48:54,927 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 60/73 batches (6000 baskets), found 5935 cointegrated so far
2025-11-06 13:48:57,058 - hypothesis_testing.cointegration.utils_parallel - I

Cointegration rate (after dedup): 3.1%


## Step 5: Filter by Sustainability

Test cointegration persistence across rolling windows and discrete periods using **cointegration data**.
This ensures the cointegration relationship is stable over time (not just in-sample).

In [7]:
# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 30
STEP_DAYS = 10
PERIOD_DAYS = 30

# Mean reversion filter thresholds
HALF_LIFE_THRESHOLD_DAYS = 30.0  # Maximum half-life in days

sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=cointegration_data,  # Use cointegration_data
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=MAX_WORKERS
)

logger.info(
    f"Filtered to {len(sustainable_baskets)} sustainable baskets "
    f"(from {len(cointegrated_baskets)} cointegrated after dedup)"
)

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"Sample sustainable basket: {sample['basket']}")
    print(
        f"Rolling windows persistence: "
        f"{sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}"
    )
    print(
        f"Discrete periods persistence: "
        f"{sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}"
    )


2025-11-06 13:48:58,366 - hypothesis_testing.cointegration.filter_sustainability - INFO - Testing 226 baskets for sustainability (parallelized at basket level with 34 workers)...
2025-11-06 13:49:04,562 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 11/226 baskets, found 11 sustainable so far
2025-11-06 13:49:04,914 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 22/226 baskets, found 22 sustainable so far
2025-11-06 13:49:05,221 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 33/226 baskets, found 33 sustainable so far
2025-11-06 13:49:07,010 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 44/226 baskets, found 44 sustainable so far
2025-11-06 13:49:07,443 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 55/226 baskets, found 55 sustainable so far
2025-11-06 13:49:07,718 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 6

Sample sustainable basket: ['ETH', 'ETC', 'ENS', 'AAVE', 'JTO', 'FLUX']
Rolling windows persistence: 100.0%
Discrete periods persistence: 100.0%


## Step 6: Filter by Mean Reversion Speed

Filter for baskets that mean revert quickly using **half-life data** (separate from cointegration data).
This tests whether the spread reverts fast enough to be profitable on unseen data.

In [9]:
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    test_data=test_data,  # Use test_data (prevents data leakage)
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

2025-11-06 13:51:21,002 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Testing 226 baskets for mean reversion speed (using separate test data: 13976 samples)...
2025-11-06 13:52:37,801 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 10/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:52:52,844 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 20/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:53:02,794 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 30/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:53:58,186 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 40/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:54:11,958 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 50/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:54:24,703 - hypothesis_testing.cointegration.filter


Sample fast mean-reverting basket: ['GRT', 'NEAR', 'RENDER', 'ICP', 'FIL', 'SAND']
Half-life: inf days
ADF p-value: 0.0006
Is stationary: True


## Step 7: Save Validated Baskets

Save the proven baskets for strategy deployment.

In [11]:
fast_mean_reverting_baskets

[{'basket': ['GRT', 'NEAR', 'RENDER', 'ICP', 'FIL', 'SAND'],
  'johansen_result': {'trace_stat': np.float64(85.68210967342617),
   'trace_crit': np.float64(83.9383),
   'p_value': np.float64(2.220446049250313e-16),
   'eigenvalues': array([2.64966405e-03, 1.56845014e-03, 1.14407901e-03, 4.24076472e-04,
          3.39563010e-04, 1.54703461e-07]),
   'eigenvectors': array([[ 14.08627253, -18.18788483,  -7.21157069,   1.93079058,
             3.17190396,   0.82437975],
          [-31.35536281,  -2.47765505,   6.7474996 ,   3.20262939,
            -3.65439622,  -0.89415633],
          [  3.13441584,  -1.29442789,  -4.8346367 ,   4.30165436,
             3.99754741,   2.67771715],
          [  4.20244343,  -0.39827643,  -8.11424932,  -8.96050883,
             9.23067855,  -0.45426383],
          [ 30.41415835,  -1.29616089,   2.36893688,   3.86551058,
           -13.69086324,  -1.77385645],
          [-16.32692364,  29.31616804,   4.28120575,  -5.3610179 ,
            -2.76669538,  -2.13731

In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': float(basket_result['johansen_result']['p_value']),
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': float(basket_result['sustainability']['rolling_windows']['persistence_ratio']),
        'sustainability_discrete': float(basket_result['sustainability']['discrete_periods']['persistence_ratio']),
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'adf_p_value': float(basket_result['mean_reversion']['adf_p_value']),
        'is_stationary': bool(basket_result['mean_reversion']['is_stationary'])
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets: {len(candidate_baskets)}")
print(f"Cointegrated baskets (deduplicated): {len(cointegrated_baskets)} ({len(cointegrated_baskets)/len(candidate_baskets):.1%})")
print(f"Sustainable baskets: {len(sustainable_baskets)} ({len(sustainable_baskets)/len(cointegrated_baskets):.1%} of cointegrated)")
print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({len(fast_mean_reverting_baskets)/len(sustainable_baskets):.1%} of sustainable)")
print(f"{'='*60}")
print(f"Validated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")


2025-11-06 14:07:21,954 - __main__ - INFO - Saved 2 validated baskets to validated_baskets.json


VALIDATION SUMMARY
Total candidate baskets: 7293
Cointegrated baskets (deduplicated): 226 (3.1%)
Sustainable baskets: 226 (100.0% of cointegrated)
Fast mean-reverting baskets: 2 (0.9% of sustainable)
Validated baskets ready for strategy deployment!
Output file: validated_baskets.json
